In [ ]:
## Set notebook to auto reload updated modules
%load_ext autoreload
%autoreload 2

In [ ]:
import settings, setup
from domain import uastring_domain
import ua_scraper
import ua_scraper.constants
import time
import json

In [ ]:
LOGGING_INITIALIZED = False

In [ ]:

if not LOGGING_INITIALIZED:
    setup.setup_loguru_logging(log_level="DEBUG")
    LOGGING_INITIALIZED = True

In [ ]:
import pandas as pd
import bs4

In [ ]:
soup: bs4.BeautifulSoup = ua_scraper.get_soup(url=ua_scraper.constants.ALL_UA_URL)

In [ ]:
with open("./all_ua_strings.html", "w") as f:
    f.write(soup.prettify())

In [ ]:
# ua_strings: list[str] = ua_scraper.extract_ua_strings(soup=soup)
# display(len(ua_strings))

In [ ]:
## Extract the 'liste' div that has all the user agents
list_div: bs4.Tag = soup.find("div", attrs={"id": "liste"})
len(list_div)

In [ ]:
categories: bs4.ResultSet[bs4.Tag] = list_div.find_all("h3")
len(categories)

In [ ]:
soup_sections: list[dict] = []

In [ ]:
link_lists: list[bs4.Tag] = list_div.find_all("ul")
display(len(link_lists))

li_paths = []
for link_list in link_lists:
    link_list_items = link_list.find_all("li")
    li_paths = li_paths + link_list_items
display(len(li_paths))

In [ ]:
links: list[bs4.Tag] = []
for li_path in li_paths:
    link_items: list[bs4.Tag] = li_path.find_all("a")
    links = links + link_items
display(len(links))
display(links[:5])

In [ ]:
ua_strings: list[bs4.Tag] = []
for link in links:
    ua_strings.append(link.text)
    
display(len(ua_strings))

In [ ]:
with open("ua_strings.txt", "w") as f:
    f.write("\n".join(ua_strings))

In [ ]:
for category in categories:
    category_name = category.text
    # display(f"Category: {category_name}")
    
    section = {"category": category_name, "links": []}

---

Alternative way

In [ ]:
user_agents = []

for link_list in list_div.find_all("ul"):
    for li_path in link_list.find_all("li"):
        for link in li_path.find_all("a"):
            user_agents.append(link)

display(len(user_agents))

---

In [ ]:
ua_category_links: dict = ua_scraper.scrape_ua_categories()

In [ ]:
display(ua_category_links["links"][:5])
display(ua_category_links["extracted_tags"][:5])

In [ ]:
with open("ua_category_links.json", "w") as f:
    _data = json.dumps(ua_category_links, indent=4, sort_keys=True, default=str)
    f.write(_data)

In [ ]:
ua_strings = []

In [ ]:
category_soups: list[bs4.BeautifulSoup] = []

In [ ]:
len(ua_category_links["links"])

In [ ]:
# category_uas: list[dict] = []

# for category in ua_category_links["links"][:5]:
#     display(category)
#     try:
#         category_soup = ua_scraper.get_soup(url=category["link"])
#     except Exception as exc:
#         msg= f"({type(exc)}) Error scraping page '{category['link']}'. Details: {exc}"
#         display(f"[ERROR] {msg}")
#         continue

#     ua_strings = ua_scraper.extract_ua_strings(soup=category_soup)
#     category_uas.append({"client": category["name"], "user_agents": ua_strings})

#     time.sleep(2)
    
# # display(category_uas[:5])

In [ ]:
category_uas =  ua_scraper.crawl_ua_categories()

In [ ]:
len(category_uas)

In [ ]:
with open("categorized_uas.json", "w") as f:
    _data = json.dumps(category_uas, indent=4, sort_keys=True, default=str)
    f.write(_data)

In [ ]:
len(category_uas)

In [ ]:
ua_schemas: list[uastring_domain.UAPageScrapeIn] = []

for d in category_uas:
    _schema = uastring_domain.UAPageScrapeIn(client=d["client"], user_agents=d["user_agents"])
    ua_schemas.append(_schema)

ua_schemas[:5]